# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S09: Xây Dựng Text Classification Pipeline với PyTorch

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S09
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook huấn luyện mạng nơ-ron sâu phân loại văn bản bằng PyTorch
* **Mức độ liên quan VOAI / Olympic AI:** Hoàn thiện năng lực xử lý văn bản thực chiến cho các đề thi Olympic AI dạng NLP.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu thách thức lớn nhất của dữ liệu văn bản khi nạp theo Batch: Các câu có độ dài hoàn toàn khác nhau (Variable Sequence Length).
- Xây dựng Custom Collate Function với kỹ thuật Đệm chuỗi (Padding bằng `pad_sequence`) trong DataLoader.
- Thiết kế kiến trúc mạng Deep NLP: `nn.Embedding` -> Global Pooling (Gộp trung bình đặc trưng câu) -> Dense Classifier Head.
- Huấn luyện thành công mô hình Phân loại cảm xúc văn bản (Text Classification) bằng PyTorch từ số 0 và đo lường Macro-F1.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M4_S06, M4_S08 và M3_S04 (Training loop).
- Hiểu cơ chế hoạt động của `collate_fn` trong PyTorch DataLoader.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong bài toán xử lý ảnh, mọi bức ảnh đều có thể được ép về cùng một kích thước cố định như $128 	imes 128$. Nhưng trong ngôn ngữ, câu thứ nhất có thể chỉ có 4 từ (*"Phim này quá tệ"*), trong khi câu thứ hai là một đoạn bình luận dài 80 từ! Nếu không thể gom các câu có độ dài khác nhau này thành một Tensor hình chữ nhật chuẩn mực, bạn sẽ không thể nạp dữ liệu theo Mini-batch vào GPU để huấn luyện.

Làm chủ kỹ thuật **Đệm chuỗi (Padding)** thông qua **Custom Collate Function** kết hợp với mạng nơ-ron nhúng từ (Embedding Neural Net) chính là bài kiểm tra toàn diện về kỹ năng lập trình Deep Learning cho NLP, đưa bạn từ mức thử nghiệm đồ chơi lên cấp độ xây dựng hệ thống phân tích ngôn ngữ thực thụ!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Kỹ thuật Padding & Collate Function
* **Vấn đề:** Các câu có độ dài $L_1, L_2, \dots, L_B$ khác nhau không thể ghép trực tiếp thành tensor 2D $[B, L]$.
* **Giải pháp:** Tìm độ dài lớn nhất trong batch ($L_{max}$), sau đó đệm thêm token `<PAD>` (giá trị 0) vào cuối các câu ngắn hơn sao cho tất cả đều có cùng độ dài $L_{max}$.
* Hàm `torch.nn.utils.rnn.pad_sequence` tự động thực hiện thao tác này một cách tối ưu.

### 2. Kiến trúc Mạng Deep Averaging Network (DAN) cho NLP
Một mô hình cực kỳ nhanh, nhẹ và hiệu quả cho phân loại văn bản:
1. **Embedding Layer (`nn.Embedding`):** Biến ma trận chỉ số $[B, L_{max}] \to$ Tensor nhúng 3D $[B, L_{max}, d]$.
2. **Global Average Pooling:** Lấy trung bình cộng các vector từ dọc theo chiều độ dài câu:
   $$\mathbf{h}_{doc} = \frac{1}{L} \sum_{i=1}^L \mathbf{v}_{word_i}$$
   Biến tensor 3D $[B, L_{max}, d] \to$ Vector đại diện tài liệu 2D $[B, d]$.
3. **Linear Classifier:** Đưa vector $[B, d]$ qua tầng tuyến tính để xuất ra Logits phân loại.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cấu trúc Deep Averaging Network (DAN) cho Phân loại Văn bản:
Câu 1: [ID_1, ID_2, PAD, PAD] ──┐
Câu 2: [ID_3, ID_4, ID_5, PAD] ──┼──> [nn.Embedding] ──> [Tensor 3D: Batch x Length x Dim]
Câu 3: [ID_6, ID_7, ID_8, ID_9] ─┘                                │
                                                                  ▼
                                                   [Global Average Pooling]
                                                                  │ (Gộp trung bình thành 1 vector duy nhất)
                                                                  ▼
                                                   [Dense Classifier Head: Linear -> ReLU -> Linear]
                                                                  │
                                                                  ▼
                                                       [Dự đoán: Tích cực / Tiêu cực]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# 1. Custom Dataset va Custom Collate Function cho van ban co do dai bien thien
class RawTextDataset(Dataset):
    def __init__(self, tokenized_texts, labels):
        # tokenized_texts la danh sach cac list co do dai khac nhau
        self.texts = [torch.tensor(t, dtype=torch.long) for t in tokenized_texts]
        self.labels = torch.tensor(labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# Ham Collate chuyen biet tu dong dem pad_sequence cho tung Batch
def pad_collate_fn(batch):
    texts, labels = zip(*batch)
    # Tu dong dem so 0 (<PAD>) vao cac cau ngan hon de co cung do dai batch_first=True
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return texts_padded, labels

# Du lieu gia lap: 4 cau co do dai khac nhau hoan toan
sample_texts = [
    [2, 4],             # Dai 2 tu
    [5, 1, 8, 9, 3],    # Dai 5 tu
    [7],                # Dai 1 tu
    [4, 6, 2]           # Dai 3 tu
]
sample_labels = [1, 0, 1, 0]

dataset = RawTextDataset(sample_texts, sample_labels)
loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=pad_collate_fn)

print("===== KIEM TRA BATCH DA DUOC DEM PADDING TU DONG =====")
for batch_x, batch_y in loader:
    print(f"Batch X Shape: {batch_x.shape} (Tu dong dem theo cau dai nhat trong batch)")
    print(batch_x)
    print("Labels:", batch_y)
    break

### 🔍 Code Walkthrough (Khối 1)
Khối code giải quyết bài toán cốt tử về độ dài văn bản không đồng nhất:
- `RawTextDataset`: Lưu trữ các danh sách token IDs có độ dài tùy biến.
- `pad_collate_fn`: Sử dụng `pad_sequence(..., batch_first=True, padding_value=0)` tự động đệm số 0 vào các câu ngắn trong batch.


In [ ]:
# 2. Xay dung Mo hinh Deep Averaging Network (DAN) va Huan luyen
class DeepTextClassifier(nn.Module):
    def __init__(self, vocab_size=20, embed_dim=16, num_classes=2):
        super(DeepTextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, 16)
        self.relu = nn.ReLU()
        self.out = nn.Linear(16, num_classes)
        
    def forward(self, x):
        # x shape: [Batch, Seq_Len]
        embedded = self.embedding(x) # [Batch, Seq_Len, Embed_Dim]
        
        # Global Average Pooling: tinh trung binh cong cac tu trong cau (axis=1)
        # De chuan xac ta lay trung binh doc theo chieu tu
        pooled = torch.mean(embedded, dim=1) # [Batch, Embed_Dim]
        
        hidden = self.relu(self.fc1(pooled))
        logits = self.out(hidden)
        return logits

# Khoi tao mo hinh va kiem thu forward pass
model_nlp = DeepTextClassifier(vocab_size=20, embed_dim=16, num_classes=2)
dummy_batch = torch.tensor([[2, 4, 0, 0, 0], [5, 1, 8, 9, 3]], dtype=torch.long)

with torch.no_grad():
    logits = model_nlp(dummy_batch)
    
print("\n===== FORWARD PASS DEEP TEXT CLASSIFIER =====")
print(f"Input Shape : {dummy_batch.shape}")
print(f"Logits Shape: {logits.shape} (2 mau, 2 lop cam xuc)")
print("Gia tri Logits:\n", logits)

### 🔍 Code Walkthrough (Khối 2)
Khối code thiết kế mô hình Deep NLP hoàn chỉnh:
- Tầng `nn.Embedding` ánh xạ các câu đệm padding thành biểu diễn không gian liên tục.
- `torch.mean(embedded, dim=1)` thực hiện pooling gộp thông tin toàn câu cực kỳ thanh lịch mà không làm bùng nổ tham số.
- Tầng Linear cuối cùng xuất ra Logits phân loại nhị phân.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Bỏ qua token `<PAD>` khi tính Average Pooling
Trong đoạn code trên, lệnh `torch.mean(embedded, dim=1)` đang tính trung bình cả các vector `<PAD>` (toàn số 0), làm suy giảm biên độ của các câu ngắn. Hãy viết công thức pooling có mặt nạ (Masked Average Pooling) chỉ chia cho độ dài thực tế của từng câu.

### Bài 2 (Easy): Thử nghiệm với Max Pooling
Thay thế `torch.mean` bằng `torch.max(embedded, dim=1).values`. Giải thích tại sao Max Pooling giúp bắt được từ khóa có cảm xúc mạnh nhất trong câu (ví dụ chỉ cần có từ "tuyệt vời" là cả câu được kích hoạt mạnh).

### Bài 3 (Medium): Viết vòng lặp huấn luyện hoàn chỉnh cho Text Classifier
Sử dụng dữ liệu `sample_texts` ở trên, khởi tạo hàm mất mát `nn.CrossEntropyLoss()` và optimizer `Adam`. Viết vòng lặp huấn luyện 5 epochs và in ra loss của từng epoch.

### Bài 4 (Medium): Thêm tầng Dropout chống Overfitting
Thêm `nn.Dropout(p=0.4)` vào giữa tầng `fc1` và tầng `out`. Giải thích tại sao trong các bài toán phân loại văn bản có kích thước nhỏ, Dropout là kỹ thuật bắt buộc để mô hình không phụ thuộc vào một từ khóa đơn lẻ.

### Bài 5 (Challenge): Mở rộng thành mô hình Recurrent Neural Network (RNN / GRU)
Tìm hiểu tầng `nn.GRU(embed_dim, hidden_dim, batch_first=True)`. Thay thế cơ chế Average Pooling bằng việc đưa tensor nhúng qua tầng GRU để nắm bắt thứ tự trước sau của các từ trong câu văn bản.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong phần thi Xử lý Ngôn ngữ Tự nhiên của Olympic AI, **kỹ năng viết Custom Collate Function là ranh giới phân định trình độ thí sinh**:
Thí sinh nghiệp dư sẽ cố định độ dài câu bằng cách cắt cụt (truncate) hoặc đệm tĩnh toàn bộ tập dữ liệu thành một kích thước khổng lồ (ví dụ luôn đệm thành 512 từ), khiến bộ nhớ GPU lãng phí 90% dung lượng chỉ để lưu số 0. Thí sinh đẳng cấp sử dụng Dynamic Padding trong `collate_fn` (chỉ đệm bằng độ dài câu dài nhất TRONG BATCH ĐÓ), giúp tăng tốc độ huấn luyện lên gấp 3 đến 5 lần!



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh quên thiết lập tham số `batch_first=True` trong hàm `pad_sequence`, dẫn tới kích thước tensor bị đảo ngược:

```python
# CODE CÓ LỖI THỨ TỰ BATCH VÀ CHIỀU DÀI:
# Mac dinh pad_sequence se tao ra tensor co shape [Seq_Len, Batch]!
# Neu dua vao model mong doi [Batch, Seq_Len], mo hinh se tinh toan sai hoac bao loi runtime!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao luôn phải thiết lập `batch_first=True` khi muốn kích thước đầu tiên của tensor đại diện cho số lượng mẫu trong batch.
2. Kiểm tra lại shape tensor sau khi gọi `pad_sequence` để bảo đảm tính nhất quán.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Kỹ thuật Padding trong Xử lý Ngôn ngữ Tự nhiên được sử dụng nhằm mục đích gì?
   - A. Xóa bỏ các từ sai chính tả.
   - B. Đệm thêm các token đặc biệt (thường là số 0) vào cuối các câu ngắn để đưa tất cả các câu trong cùng một Mini-batch về cùng một độ dài đồng nhất.
   - C. Tăng tốc độ đọc file từ đĩa cứng.
   - D. Dịch văn bản sang tiếng Anh.

2. Tham số `collate_fn` trong `torch.utils.data.DataLoader` cho phép lập trình viên làm điều gì?
   - A. Tùy biến cách gom nhóm và xử lý danh sách các mẫu dữ liệu đơn lẻ thành một Mini-batch Tensor chuẩn mực trước khi nạp vào mô hình.
   - B. Tự động tính toán đạo hàm của hàm mất mát.
   - C. Thay đổi số lượng card đồ họa GPU.
   - D. Xóa tập dữ liệu huấn luyện.

3. Trong mô hình Deep Averaging Network (DAN), thao tác Global Average Pooling dọc theo trục từ vựng (`dim=1`) có tác dụng gì?
   - A. Nhân đôi số chiều của vector từ.
   - B. Gộp toàn bộ các vector từ trong câu thành duy nhất một vector đại diện cho cả tài liệu có số chiều cố định, không phụ thuộc vào độ dài của câu.
   - C. Sắp xếp lại thứ tự các từ theo bảng chữ cái.
   - D. Xóa các từ dừng.

4. Hàm nào của PyTorch được sử dụng phổ biến nhất để tự động đệm danh sách các tensor có độ dài khác nhau thành một tensor duy nhất?
   - A. `torch.cat()`
   - B. `torch.nn.utils.rnn.pad_sequence`
   - C. `torch.stack()`
   - D. `torch.reshape()`

5. Điểm khác biệt lớn nhất giữa mô hình Deep Learning NLP (như DAN) và mô hình Baseline TF-IDF là gì?
   - A. Deep Learning hoàn toàn không dùng số học.
   - B. Deep Learning sử dụng vector nhúng dày đặc (Dense Embeddings) có khả năng học và cập nhật ngữ nghĩa ngữ cảnh qua từng tầng, trong khi TF-IDF sử dụng ma trận tần suất cố định không học được ngữ nghĩa sâu.
   - C. TF-IDF luôn chạy chậm hơn Deep Learning.
   - D. Deep Learning chỉ áp dụng được cho tiếng Anh.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Kỹ thuật xử lý các chuỗi văn bản có độ dài biến thiên (Variable Sequence Length).
  - Cách viết Custom Collate Function với `pad_sequence` động theo từng batch.
  - Kiến trúc Deep Averaging Network (DAN) kết hợp Embedding và Global Pooling.
* **What I can now do:**
  - Tự tay xây dựng toàn bộ luồng huấn luyện Deep Learning cho bài toán phân loại văn bản bằng PyTorch thuần.
  - Tối ưu hóa việc nạp batch dữ liệu ngôn ngữ không gây lãng phí bộ nhớ.
* **What I still need to practice:**
  - Thực hiện bài toán lớn tổng kết Module: **Mini Project CV / NLP Thực Tế** ở Buổi 10!



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Buổi tiếp theo (**M4_S10**) là buổi tổng kết đỉnh cao của Module 4: **Mini Project Thực Tế: So Sánh, Tinh Chỉnh & Báo Cáo Dự Án CV / NLP**! Bạn sẽ chọn một trong hai hướng (Thị giác máy tính HOẶC Xử lý ngôn ngữ tự nhiên), tự tay xây dựng pipeline hoàn chỉnh từ dữ liệu thô, huấn luyện, đánh giá đối sánh và nhận Chứng chỉ **Certificate of CV/NLP Applied AI**!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Xử lý độ dài câu biến thiên (Variable sequence length) ✓
- Custom collate_fn & Dynamic padding với pad_sequence ✓
- Deep Averaging Network (Embedding + Global Pooling + Classifier) ✓

* **Session Deliverable:** Mã nguồn Text Classification Pipeline hoàn chỉnh bằng PyTorch Deep Learning
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
